In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, element_at, when

HOST_IP = "192.168.64.1"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .getOrCreate()
)

print("Spark Session created successfully.")

prod_a_topic = "camera-events-A"

topic_stream_df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", f"{HOST_IP}:9092")
    .option("subscribe", prod_a_topic)
    .load()
)

topic_stream_df.printSchema()

output_stream_df = (
    topic_stream_df
    .select(                                      # 1
        split(
            topic_stream_df.value.cast('string'),
            ':'
        )
        .alias('data')
    )
    .withColumn('data', element_at('data', 2))    # 2
    .withColumn(
        'data',
        (
            when( col('data') == '', '*' )        # 3A
            .otherwise( col('data') )             # 3B
        )
    )
)

output_stream_df.printSchema()

console_logger = (
    output_stream_df
    .writeStream
    .outputMode('append')
    .format('console')
)

writer = console_logger

try:
    query = writer.start()
    query.awaitTermination()
except KeyboardInterrupt:
    print("Stopping the streaming query...")
    query.stop()
except StreamingQueryException as e:
    print(f"Streaming query error: {e}")
finally:
    spark.stop()

Spark Session created successfully.
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

root
 |-- data: string (nullable = true)



ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
RuntimeError: reentrant call inside <_io.BufferedReader name=51>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/conda/lib/python3.8/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.8/soc

NameError: name 'StreamingQueryException' is not defined